# Lesson 09 - Human approval for consequential tools

**Level:** advanced | **Demo time:** 12 minutes | **Builds on:** Lessons 02 and 08

A model can propose a consequential action, but the application must decide whether that action is allowed to run. In this lesson, a Contoso refund agent pauses before a simulated refund tool, exposes the exact proposed function call for review, and resumes only after an explicit human decision.

## Learning objectives

- Mark a consequential tool with `approval_mode="always_require"`.
- Inspect a workflow's structured approval request before execution.
- Validate an exact `Approve` or `Reject` decision and resume the paused workflow.
- Prove that rejection does not execute the tool while approval does.
- Separate authentication, authorization, audit, and checkpoint responsibilities.

## Approval mental model

```text
Customer request
  -> refund agent proposes a function call
  -> workflow emits request_info(function_approval_request)
  -> application shows the exact tool name and arguments
  -> authorized human chooses Approve or Reject
  -> application resumes with a structured approval response
  -> tool body runs only when approved
```

Approval is an application-owned control. The model proposes an action; it does not authenticate the reviewer, grant permissions, or prove that a refund is valid.

In [ ]:
from collections.abc import AsyncIterable
from typing import Annotated

from agent_framework import Agent, AgentResponseUpdate, Content, WorkflowEvent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder
from pydantic import Field

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
credential = create_credential()
chat_client: FoundryChatClient = create_chat_client(settings, credential)

## Make execution observable without a real refund

The tool below changes only notebook memory. Its audit list and counter are updated **inside the tool body**, so they change only if MAF actually invokes the approved function call. Rejection therefore has a concrete, inspectable result: the counter and audit list stay unchanged.

The amount and order checks also illustrate an important boundary: approval does not replace validation or business rules inside the trusted operation.

In [ ]:
refund_audit: list[dict[str, str | float]] = []
refund_execution_count = 0
simulated_refundable_amounts = {"ORD-1001": 249.0}


@tool(approval_mode="always_require")
def issue_simulated_refund(
    order_id: Annotated[str, Field(description="Order ID to refund, for example ORD-1001.")],
    amount_usd: Annotated[
        float,
        Field(gt=0, le=500, description="Refund amount in US dollars."),
    ],
    reason: Annotated[str, Field(description="Concise reason for the refund.")],
) -> str:
    """Issue a consequential but simulated refund after human approval."""
    global refund_execution_count

    normalized_order_id = order_id.strip().upper()
    refundable_amount = simulated_refundable_amounts.get(normalized_order_id)
    if refundable_amount is None:
        return f"Refund not issued: {normalized_order_id} is not refundable in this simulation."
    if amount_usd != refundable_amount:
        return (
            f"Refund not issued: expected ${refundable_amount:.2f} for {normalized_order_id}, "
            f"not ${amount_usd:.2f}."
        )

    refund_execution_count += 1
    refund_id = f"RF-{refund_execution_count:04d}"
    refund_audit.append(
        {
            "refund_id": refund_id,
            "order_id": normalized_order_id,
            "amount_usd": amount_usd,
            "reason": reason,
            "status": "simulated",
        }
    )
    return f"Simulated refund {refund_id} issued for {normalized_order_id}: ${amount_usd:.2f}."

## Create the refund specialist

The agent uses the workshop's existing `FoundryChatClient` helper and receives only the refund capability. The tool's decorator supplies the approval policy; `SequentialBuilder` needs no special approval configuration.

Each demonstration run builds a fresh workflow so the rejection and approval paths do not share orchestration state.

In [ ]:
refund_agent = Agent(
    client=chat_client,
    name="refund_agent",
    description="Handles verified, simulated customer refunds that require human approval.",
    instructions=(
        "You are a Contoso refund specialist. For the workshop request, call "
        "issue_simulated_refund exactly once with order_id ORD-1001, amount_usd 249.0, "
        "and a concise damaged-item reason. Never claim a refund was issued unless the tool "
        "returns success. If approval is rejected, do not retry the tool and clearly state that "
        "no refund was issued."
    ),
    tools=[issue_simulated_refund],
)

## Inspect, decide, and resume

A required tool approval arrives as a `WorkflowEvent` whose type is `request_info`. Its `data` is a `Content` item of type `function_approval_request`. The application displays the nested function call, collects an exact decision, and maps the event's `request_id` to `request.to_function_approval_response(...)`.

Because the workflow is streaming, each `output` event contains one `AgentResponseUpdate` chunk rather than a list of messages. Invalid input is rejected explicitly. The workflow is then resumed with `workflow.run(stream=True, responses=responses)` until it completes.

In [ ]:
def prompt_for_approval() -> bool:
    while True:
        decision = input("Type exactly Approve or Reject: ")
        if decision == "Approve":
            return True
        if decision == "Reject":
            return False
        print("Invalid input. Enter exactly Approve or Reject, including capitalization.")


async def inspect_workflow_events(
    stream: AsyncIterable[WorkflowEvent],
) -> dict[str, Content] | None:
    responses: dict[str, Content] = {}

    async for event in stream:
        if event.type == "request_info":
            if not isinstance(event.data, Content):
                raise TypeError("Expected request_info event data to be Content.")

            request = event.data
            if request.type != "function_approval_request":
                raise ValueError(f"Unsupported request_info content type: {request.type}")
            if request.function_call is None:
                raise ValueError("Approval request did not include a function call.")

            function_call = request.function_call
            print("\n[APPROVAL REQUIRED]")
            print("Function:", function_call.name)
            print("Arguments:", function_call.arguments)
            approved = prompt_for_approval()
            responses[event.request_id] = request.to_function_approval_response(
                approved=approved
            )
            print("Decision recorded:", "Approve" if approved else "Reject")

        elif event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                raise TypeError("Expected streaming output event data to be AgentResponseUpdate.")

            update = event.data
            if update.text:
                speaker = update.author_name or update.role or "agent"
                print(f"[{speaker}]: {update.text}")

    return responses or None


async def run_refund_scenario(label: str) -> int:
    workflow = SequentialBuilder(participants=[refund_agent]).build()
    executions_before = refund_execution_count

    print(f"\n=== {label} ===")
    prompt = (
        "Order ORD-1001 arrived damaged. Issue the full simulated refund of $249.00 now."
    )
    responses = await inspect_workflow_events(workflow.run(prompt, stream=True))

    while responses is not None:
        responses = await inspect_workflow_events(
            workflow.run(stream=True, responses=responses)
        )

    executions_after = refund_execution_count
    execution_delta = executions_after - executions_before
    print("Simulated refund executions in this run:", execution_delta)
    print("Audit entries now:", refund_audit)
    return execution_delta

## Run both decision paths

This cell intentionally pauses twice:

1. Type exactly **`Reject`** for the first request. The execution delta must remain `0`, and the audit list must stay empty.
2. Type exactly **`Approve`** for the second request. The execution delta must become `1`, and the audit list must contain one simulated refund.

The assertions turn the teaching claim into a checked outcome.

In [ ]:
refund_audit.clear()
refund_execution_count = 0

print("First decision: enter Reject")
rejected_execution_delta = await run_refund_scenario("Rejection path")
assert rejected_execution_delta == 0, "Rejection must not execute the refund tool."
assert refund_audit == [], "Rejection must leave the simulated audit list empty."

print("\nSecond decision: enter Approve")
approved_execution_delta = await run_refund_scenario("Approval path")
assert approved_execution_delta == 1, "Approval should execute the refund tool exactly once."
assert len(refund_audit) == 1, "Approval should create one simulated audit entry."

print("\nVerified: rejection caused no side effect; approval caused one simulated side effect.")

## Production boundaries

| Boundary | Production responsibility |
|---|---|
| **Authentication** | Authenticate the customer and the approver outside the model. Bind the approval UI and resume request to trusted identities and tenants. |
| **Authorization** | Re-evaluate permission at decision time and again inside the refund service. Check order ownership, amount limits, approver role, policy, and current transaction state. Approval is not a substitute for authorization. |
| **Audit** | Persist an append-only record of request ID, proposed tool and arguments, approver identity, decision, timestamp, correlation ID, and final tool outcome. The notebook list is only an execution probe. Avoid logging unnecessary sensitive data. |
| **Checkpoint and resume** | This demo keeps the paused workflow in memory. For delayed approval, use durable checkpoint storage, encrypt state, apply retention, bind the checkpoint to the authorized caller, and resume only the matching outstanding request. A checkpoint restores orchestration state; it does not grant permission. |
| **Business side effect** | Execute through an authenticated, idempotent service with transaction controls. Re-check invariants immediately before the write so stale or replayed approvals cannot issue duplicate refunds. |

> Production rule: the approval response authorizes one reviewed proposal, not arbitrary future calls to the same tool.

## Tool approval is not handoff free-form input

Lesson 06 used a handoff request to collect a customer's next natural-language message. That **free-form input** advances a conversation and may clarify intent, provide missing facts, or redirect a specialist.

A tool approval request is different: it is a **structured decision about one concrete function call** whose name and arguments are already available for review. Do not treat a conversational "yes" as approval, and do not replace the structured response with prompt text. Production interfaces should show the exact proposed action and bind the approver's decision to that request ID.

In [ ]:
credential.close()

## Recap

MAF turns a consequential tool call into an inspectable pause and a structured resume. The application still owns identity, authorization, durable state, audit evidence, and the real transaction. With those boundaries explicit, an agent can propose useful actions without being trusted to approve its own work.